In [2]:
import pandas as pd
import numpy as np
from pathlib import Path



In [22]:


DATA_DIR = Path("../data/processed")

features_path = DATA_DIR / "products_features.csv"
content_path = DATA_DIR / "content_recommendations.csv"
knowledge_path = DATA_DIR / "knowledge_recommendations.csv"

for path in [features_path, content_path, knowledge_path]:
    print(path, "->", "trouvé" if path.exists() else "absent")

..\data\processed\products_features.csv -> trouvé
..\data\processed\content_recommendations.csv -> trouvé
..\data\processed\knowledge_recommendations.csv -> trouvé


In [23]:
required_paths = [features_path, content_path, knowledge_path]
missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Fichiers absents : " + ", ".join(missing_paths)
        + ". Exécute d'abord les notebooks 03, 04 et 05."
    )

products = pd.read_csv(features_path, encoding="utf-8-sig")
content = pd.read_csv(content_path, encoding="utf-8-sig")
knowledge = pd.read_csv(knowledge_path, encoding="utf-8-sig")

print("Produits :", products.shape)
print("Candidats Content-Based :", content.shape)
print("Relations Knowledge-Based :", knowledge.shape)


Produits : (9, 23)
Candidats Content-Based : (45, 8)
Relations Knowledge-Based : (0, 12)


In [20]:
required_products = {"product_id", "name"}
required_content = {"query_product_id", "product_id", "similarity_score"}
required_knowledge = {"product_id", "related_product_id", "relation_type", "evidence_source_id"}

for label, frame, required in [
    ("products_features", products, required_products),
    ("content_recommendations", content, required_content),
    ("knowledge_recommendations", knowledge, required_knowledge),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Colonnes absentes dans {label} : {sorted(missing)}")

print("Les colonnes nécessaires sont présentes.")


Les colonnes nécessaires sont présentes.


In [24]:
products["product_id"] = products["product_id"].astype(str).str.strip()
content["query_product_id"] = content["query_product_id"].astype(str).str.strip()
content["product_id"] = content["product_id"].astype(str).str.strip()
knowledge["product_id"] = knowledge["product_id"].astype(str).str.strip()
knowledge["related_product_id"] = knowledge["related_product_id"].astype(str).str.strip()

content["similarity_score"] = pd.to_numeric(
    content["similarity_score"], errors="coerce"
).fillna(0).clip(0, 1)

print("Identifiants et scores normalisés.")



Identifiants et scores normalisés.


In [25]:
knowledge_candidates = knowledge.copy()

if "verification_status" in knowledge_candidates.columns:
    knowledge_candidates = knowledge_candidates[
        knowledge_candidates["verification_status"]
        .astype(str).str.strip().str.lower().eq("verified")
    ].copy()
else:
    raise ValueError(
        "La colonne verification_status est absente : "
        "impossible de confirmer que les relations sont vérifiées."
    )

knowledge_candidates = knowledge_candidates[
    knowledge_candidates["evidence_source_id"].notna()
    & knowledge_candidates["evidence_source_id"].astype(str).str.strip().ne("")
].copy()

knowledge_candidates = knowledge_candidates.rename(columns={
    "product_id": "source_product_id",
    "related_product_id": "candidate_product_id",
    "relation_type": "knowledge_relation"
})

knowledge_candidates["content_score"] = 0.0
knowledge_candidates["knowledge_verified"] = True

knowledge_candidates = knowledge_candidates[
    ["source_product_id", "candidate_product_id", "content_score",
     "knowledge_relation", "evidence_source_id", "knowledge_verified"]
]

display(knowledge_candidates.head())
print("Relations métier vérifiées retenues :", len(knowledge_candidates))


,source_product_id,candidate_product_id,content_score,knowledge_relation,evidence_source_id,knowledge_verified


Relations métier vérifiées retenues : 0


In [28]:
# Préparer les candidats des deux méthodes

content_candidates = content.copy()
knowledge_candidates = knowledge.copy()

print("Content candidates :", content_candidates.shape)
print("Knowledge candidates :", knowledge_candidates.shape)

Content candidates : (45, 8)
Knowledge candidates : (0, 12)


In [31]:
print("=== CONTENT ===")
display(content.head())

print("\n=== KNOWLEDGE ===")
display(knowledge.head())

=== CONTENT ===


,query_product_id,product_id,name,domain,category,brand,price_dzd,similarity_score
0,SH001,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,29500.0,0.154967
1,SH001,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,NaN,0.048764
2,SH001,SH008,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,24000.0,0.033456
3,SH001,SH007,accélérateur wilo 82w avec bloc pour chaudière...,chauffage,pompe de circulation,wilo,21300.0,0.029217
4,SH001,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,21800.0,0.006950



=== KNOWLEDGE ===


,product_id,related_product_id,relation_type,conditions,evidence_source_id,verification_status,source_name,target_name,target_domain,target_category,target_brand,target_model


In [32]:

import pandas as pd

# ==========================================
# 1. Préparer les candidats Content-Based
# ==========================================

content_candidates = pd.DataFrame({
    "source_product_id": content["query_product_id"],
    "candidate_product_id": content["product_id"],
    "content_score": pd.to_numeric(
        content["similarity_score"], errors="coerce"
    ),
    "knowledge_relation": "",
    "evidence_source_id": "",
    "knowledge_verified": False
})


# ==========================================
# 2. Préparer les candidats Knowledge-Based
# ==========================================

knowledge_candidates = pd.DataFrame({
    "source_product_id": knowledge["product_id"],
    "candidate_product_id": knowledge["related_product_id"],
    "content_score": float("nan"),
    "knowledge_relation": knowledge["relation_type"],
    "evidence_source_id": knowledge["evidence_source_id"],
    "knowledge_verified": (
        knowledge["verification_status"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("verified")
    )
})


# ==========================================
# 3. Combiner les deux méthodes
# ==========================================

all_candidates = pd.concat(
    [content_candidates, knowledge_candidates],
    ignore_index=True
)

print("Content-Based :", content_candidates.shape)
print("Knowledge-Based :", knowledge_candidates.shape)
print("Total :", all_candidates.shape)

display(all_candidates.head())



Content-Based : (45, 6)
Knowledge-Based : (0, 6)
Total : (45, 6)


,source_product_id,candidate_product_id,content_score,knowledge_relation,evidence_source_id,knowledge_verified
0,SH001,SH002,0.154967,,,False
1,SH001,SH003,0.048764,,,False
2,SH001,SH008,0.033456,,,False
3,SH001,SH007,0.029217,,,False
4,SH001,SH004,0.006950,,,False


In [33]:
def combine_evidence(group):

    content_score = group["content_score"].max()

    verified_rows = group[
        group["knowledge_verified"] == True
    ]

    relation = "; ".join(
        sorted(
            verified_rows["knowledge_relation"]
            .dropna()
            .astype(str)
            .unique()
        )
    )

    evidence = "; ".join(
        sorted(
            verified_rows["evidence_source_id"]
            .dropna()
            .astype(str)
            .unique()
        )
    )

    verified = not verified_rows.empty

    return pd.Series({
        "content_score": content_score,
        "knowledge_relation": relation,
        "evidence_source_id": evidence,
        "knowledge_verified": verified
    })


hybrid = (
    all_candidates
    .groupby(
        ["source_product_id", "candidate_product_id"]
    )
    .apply(combine_evidence)
    .reset_index()
)

print("Nombre de paires uniques :", len(hybrid))

display(hybrid.head(10))

Nombre de paires uniques : 45


C:\Users\GEEK\AppData\Local\Temp\ipykernel_5988\3820936001.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_evidence)


,source_product_id,candidate_product_id,content_score,knowledge_relation,evidence_source_id,knowledge_verified
0,SH001,SH002,0.154967,,,False
1,SH001,SH003,0.048764,,,False
2,SH001,SH004,0.006950,,,False
3,SH001,SH007,0.029217,,,False
4,SH001,SH008,0.033456,,,False
5,SH002,SH001,0.154967,,,False
6,SH002,SH003,0.107114,,,False
7,SH002,SH004,0.010592,,,False
8,SH002,SH005,0.010592,,,False
9,SH002,SH008,0.013753,,,False


In [34]:
CONTENT_WEIGHT = 0.70
KNOWLEDGE_WEIGHT = 0.30

# Le signal Knowledge-Based vaut 1 seulement si une relation vérifiée et sourcée existe.
hybrid["knowledge_score"] = hybrid["knowledge_verified"].astype(float)

hybrid["hybrid_score"] = (
    CONTENT_WEIGHT * hybrid["content_score"]
    + KNOWLEDGE_WEIGHT * hybrid["knowledge_score"]
)

print("Poids Content-Based :", CONTENT_WEIGHT)
print("Poids Knowledge-Based :", KNOWLEDGE_WEIGHT)
display(hybrid.sort_values("hybrid_score", ascending=False).head(10))


Poids Content-Based : 0.7
Poids Knowledge-Based : 0.3


,source_product_id,candidate_product_id,content_score,knowledge_relation,evidence_source_id,knowledge_verified,knowledge_score,hybrid_score
39,SH008,SH007,0.516071,,,False,0.0,0.361250
34,SH007,SH008,0.516071,,,False,0.0,0.361250
16,SH004,SH005,0.439348,,,False,0.0,0.307543
17,SH004,SH006,0.439348,,,False,0.0,0.307543
26,SH006,SH004,0.439348,,,False,0.0,0.307543
21,SH005,SH004,0.439348,,,False,0.0,0.307543
22,SH005,SH006,0.439348,,,False,0.0,0.307543
27,SH006,SH005,0.439348,,,False,0.0,0.307543
0,SH001,SH002,0.154967,,,False,0.0,0.108477
5,SH002,SH001,0.154967,,,False,0.0,0.108477


In [35]:
product_columns = [
    col for col in ["product_id", "name", "domain", "category", "brand", "model", "price_dzd"]
    if col in products.columns
]

source_info = products[product_columns].rename(
    columns={col: f"source_{col}" for col in product_columns if col != "product_id"}
).rename(columns={"product_id": "source_product_id"})

candidate_info = products[product_columns].rename(
    columns={col: f"candidate_{col}" for col in product_columns if col != "product_id"}
).rename(columns={"product_id": "candidate_product_id"})

hybrid = hybrid.merge(source_info, on="source_product_id", how="left")
hybrid = hybrid.merge(candidate_info, on="candidate_product_id", how="left")

display(hybrid.head())


,source_product_id,candidate_product_id,content_score,knowledge_relation,evidence_source_id,knowledge_verified,knowledge_score,hybrid_score,source_name,source_domain,source_category,source_brand,source_model,source_price_dzd,candidate_name,candidate_domain,candidate_category,candidate_brand,candidate_model,candidate_price_dzd
0,SH001,SH002,0.154967,,,False,0.0,0.108477,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,smart 12l,29500.0
1,SH001,SH003,0.048764,,,False,0.0,0.034135,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,28 kw ventouse,NaN
2,SH001,SH004,0.006950,,,False,0.0,0.004865,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,15-50,21800.0
3,SH001,SH007,0.029217,,,False,0.0,0.020452,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,accélérateur wilo 82w avec bloc pour chaudière...,chauffage,pompe de circulation,wilo,82w avec bloc,21300.0
4,SH001,SH008,0.033456,,,False,0.0,0.023419,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,82w,24000.0


In [36]:
def recommend_hybrid(product_id, top_n=5):
    product_id = str(product_id).strip()

    results = hybrid[
        hybrid["source_product_id"] == product_id
    ].copy()

    results = results.sort_values(
        ["hybrid_score", "content_score"],
        ascending=False
    ).head(top_n)

    display_columns = [
        col for col in [
            "candidate_product_id", "candidate_name",
            "candidate_domain", "candidate_category", "candidate_brand",
            "content_score", "knowledge_relation",
            "knowledge_verified", "evidence_source_id", "hybrid_score"
        ] if col in results.columns
    ]

    return results[display_columns].reset_index(drop=True)


In [37]:
if products.empty:
    print("Le catalogue est vide.")
else:
    selected_product_id = products.iloc[0]["product_id"]
    print("Produit sélectionné :", selected_product_id)
    display(recommend_hybrid(selected_product_id, top_n=5))


Produit sélectionné : SH001


,candidate_product_id,candidate_name,candidate_domain,candidate_category,candidate_brand,content_score,knowledge_relation,knowledge_verified,evidence_source_id,hybrid_score
0,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,0.154967,,False,,0.108477
1,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,0.048764,,False,,0.034135
2,SH008,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,0.033456,,False,,0.023419
3,SH007,accélérateur wilo 82w avec bloc pour chaudière...,chauffage,pompe de circulation,wilo,0.029217,,False,,0.020452
4,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,0.006950,,False,,0.004865


In [38]:
output_path = DATA_DIR / "hybrid_recommendations.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

hybrid = hybrid.sort_values(
    ["source_product_id", "hybrid_score"],
    ascending=[True, False]
)

hybrid.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Fichier créé :", output_path.exists())
print("Chemin :", output_path)
print("Nombre de recommandations :", len(hybrid))


Fichier créé : True
Chemin : ..\data\processed\hybrid_recommendations.csv
Nombre de recommandations : 45


In [39]:
print("Paires source-candidat dupliquées :",
      hybrid.duplicated(["source_product_id", "candidate_product_id"]).sum())

print("Candidats avec relation métier vérifiée :",
      int(hybrid["knowledge_verified"].sum()))

print("Candidats sans relation métier vérifiée :",
      int((~hybrid["knowledge_verified"]).sum()))

display(hybrid.head(20))


Paires source-candidat dupliquées : 0
Candidats avec relation métier vérifiée : 0
Candidats sans relation métier vérifiée : 45


,source_product_id,candidate_product_id,content_score,knowledge_relation,evidence_source_id,knowledge_verified,knowledge_score,hybrid_score,source_name,source_domain,source_category,source_brand,source_model,source_price_dzd,candidate_name,candidate_domain,candidate_category,candidate_brand,candidate_model,candidate_price_dzd
0,SH001,SH002,0.154967,,,False,0.0,0.108477,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,smart 12l,29500.0
1,SH001,SH003,0.048764,,,False,0.0,0.034135,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,28 kw ventouse,NaN
4,SH001,SH008,0.033456,,,False,0.0,0.023419,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,82w,24000.0
3,SH001,SH007,0.029217,,,False,0.0,0.020452,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,accélérateur wilo 82w avec bloc pour chaudière...,chauffage,pompe de circulation,wilo,82w avec bloc,21300.0
2,SH001,SH004,0.006950,,,False,0.0,0.004865,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,15-50,21800.0
5,SH002,SH001,0.154967,,,False,0.0,0.108477,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,smart 12l,29500.0,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,NaN,28500.0
6,SH002,SH003,0.107114,,,False,0.0,0.074980,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,smart 12l,29500.0,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,28 kw ventouse,NaN
9,SH002,SH008,0.013753,,,False,0.0,0.009627,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,smart 12l,29500.0,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,82w,24000.0
7,SH002,SH004,0.010592,,,False,0.0,0.007414,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,smart 12l,29500.0,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,15-50,21800.0
8,SH002,SH005,0.010592,,,False,0.0,0.007414,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,smart 12l,29500.0,accélérateur grundfos 15-60 110w réf. 99157090,chauffage,pompe de circulation,grundfos,15-60,21300.0
